[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Shape


## What you will be able to do

Read an array's dimensions with `ndim`, `shape` and `size`, regroup the year of readings into days and
hours with `reshape`, let `-1` work out a dimension for you, and flatten, transpose or add a
dimension when a calculation needs one of those.

## The idea

### The problem

The readings arrive as four rows of 8,760 hours, one row per station, and that suits some questions
and not others. A mean for every day wants each row cut into 365 days of 24 hours. A single figure
for the whole network wants all 35,040 readings in one run. A table to save as a file, one line per
hour with a column per station, wants the rows and the columns swapped, and some libraries want one
station's readings standing as a column instead of lying as a row.

With lists, each of those is a new nested list built by index arithmetic. Hour `h` of day `d` is
`row[d * 24 + h]`, and turning stations into columns means a loop over the hours that picks one
value from each station's list. That code is easy to get slightly wrong, a 23 where a 24 should be,
and nothing tells you when you have.

### What a shape is

> An array's **shape** is a tuple with one number for each of its dimensions, giving how many values
> lie along that dimension. **`reshape`** gives the same values a new shape whose numbers multiply to
> the same total.

### Why it works that way

An array's values sit in memory as one long run, and its shape is a description of how to read that
run as rows, or as rows of rows. `reshape` changes the description and leaves the values where they
are, so it is fast at any size and does not copy anything. The run is read with the last dimension
changing fastest. In the readings, the hours of one station follow each other, so `(4, 365, 24)`
takes the first 24 hours as Bergen's first day, the next 24 as its second, and so on, which is what
a day is.

That rule is also the trap. `(4, 24, 365)` multiplies to the same total, so NumPy accepts it, and
reads Bergen's first 365 hours as the first row of something that is not a day at all. A shape
records how many values lie along each dimension, and nothing about what they mean. Swapping rows
and columns is a different operation, the transpose `.T`, which moves values to new positions, and
`reshape(8760, 4)` in its place gives a table of the right size with the wrong values in it.

A dimension of length 1 leaves the values as they are, and changes how an array lines up with
another one. Four station means of shape (4,) and the same means as a column of shape (4, 1) are
different shapes, and the **Broadcasting** notebook depends on the difference.

### Where this shows up

scikit-learn expects its input as a two-dimensional table of rows and features, so a single column
of values is reshaped with `reshape(-1, 1)` before a model will take it, which the **scikit-learn,
Deep Dive** guide does. An image is an array of shape (height, width, 3), one layer each for red,
green and blue. **The axis Argument** notebook builds on the shapes made here, and averaging over
the hours of `(4, 365, 24)` is where daily means come from.

### What this notebook covers

- `ndim`, `shape` and `size`, for the readings and for one row of them
- days and hours with `reshape(4, 365, 24)`, and the order `reshape` reads values in
- `-1` for the one dimension you would rather not work out
- flattening with `ravel` and `flatten`, and which one copies
- swapping rows and columns with `.T`
- a dimension of length 1, with `np.newaxis` or `reshape(-1, 1)`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

hours = np.arange(6)
print(hours.reshape(2, 3))
print(hours.reshape(2, 3).shape)
```

```
[[0 1 2]
 [3 4 5]]
(2, 3)
```

Six values read as two rows of three. The first row took 0, 1 and 2, the first three values in order,
and the second row took the next three.

## Setup

Three imports, and the year of readings as lists and as an array.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained. `readings` is the same values
as one array of shape (4, 8760), with `nan` for Svalbard's silent day.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan

print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### ndim, shape and size

`ndim` counts the dimensions, `shape` gives the length of each, and `size` the number of values in
all:

In [2]:
print(readings.ndim, readings.shape, readings.size)
print(readings[0].ndim, readings[0].shape)
print(np.ndim(readings[0, 0]), np.shape(readings[0, 0]))

2 (4, 8760) 35040
1 (8760,)
0 ()


One station's row is one-dimensional, shape `(8760,)`, where the comma marks a tuple of one number.
A single reading has no dimensions at all and an empty shape.

### Days and hours with reshape

In [3]:
days = readings.reshape(4, 365, 24)

print(days.shape)
print(np.array_equal(days[0, 0], readings[0, :24]))
print(days[0, 1, 0] == readings[0, 24])
print(np.shares_memory(days, readings))

(4, 365, 24)
True
True
True


`days[0, 0]` is Bergen's first 24 hours, and `days[0, 1, 0]` the first hour of its second day, hour
24 of the year. `np.shares_memory` confirms that nothing was copied: `days` is a new description of
the same values.

### The order reshape reads values in

Here are twelve values as three rows of four and as four rows of three. In both, the last dimension
fills first:

In [4]:
print(np.arange(12).reshape(3, 4))
print(np.arange(12).reshape(4, 3))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In `(3, 4)` each row took four values in a row before the next row began. That is why `(4, 365, 24)`
makes days, since the 24 consecutive hours of a station fill one day before the next day begins.

### -1 works out one dimension

`-1` in a shape stands for whatever length makes the total come out right:

In [5]:
print(readings.reshape(4, -1, 24).shape)
print(readings.reshape(-1).shape)
print(readings.reshape(-1, 24).shape)

(4, 365, 24)
(35040,)
(1460, 24)


365 days per station, 35,040 values in one run, and 1,460 rows of 24 hours, a row for every day at
every station.

### Flattening with ravel and flatten

`ravel` and `flatten` both give the values as one row. `ravel` returns a new view of the same memory
when it can, and `flatten` always copies:

In [6]:
everything = readings.ravel()

print(everything.shape)
print(round(np.nanmean(everything), 2))
print(np.shares_memory(readings, everything), np.shares_memory(readings, readings.flatten()))

(35040,)
3.38
True False


3.38 is the mean of every reading in the network, the four stations together. Use `flatten` when the
flat version will be changed and the original must not be.

### Swapping rows and columns with .T

`.T` is the transpose, which moves the value at row `i`, column `j` to row `j`, column `i`. The readings become one row per
hour, with a column for each station:

In [7]:
by_hour = readings.T

print(by_hour.shape)
print(by_hour[0])
print(readings[:, 0])

(8760, 4)
[ -3.6  -3.5 -14.6  -6.7]
[ -3.6  -3.5 -14.6  -6.7]


Row 0 of `by_hour` is the first hour of the year at all four stations, the same values as column 0 of
`readings`. Like `reshape`, `.T` copies nothing, and only changes how the values are read.

### A dimension of length 1

`np.nanmean(readings, axis=1)` gives one mean per station, shape (4,). `np.newaxis` inside the
brackets adds a dimension of length 1 where it stands, and `reshape(-1, 1)` does the same:

In [8]:
means = np.nanmean(readings, axis=1)

print(means.shape)
print(means[:, np.newaxis].shape, means[np.newaxis, :].shape, means.reshape(-1, 1).shape)
print(means[:, np.newaxis].round(2))

(4,)
(4, 1) (1, 4) (4, 1)
[[ 8.  ]
 [ 6.5 ]
 [-4.48]
 [ 3.5 ]]


The same four means stand as a column of shape (4, 1). Taken from the readings, that column gives
every hour its difference from its own station's mean, a shape of (4, 8760), which the
**Broadcasting** notebook explains:

In [9]:
differences = readings - means[:, np.newaxis]

print(differences.shape)
print(differences[:, 0].round(2))

(4, 8760)
[-11.6  -10.   -10.12 -10.2 ]


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/04-shape-solutions.ipynb).

**1.** Reshape `np.arange(24)` into 4 rows of 6, and print it and its shape.

In [10]:
# your code here

**2.** Reshape Oslo's readings, row 1, into 365 days of 24 hours, and print the readings for 1 January.

In [11]:
# your code here

**3.** Use `-1` to reshape `readings` into rows of 24 hours, however many rows that takes, and print the shape.

In [12]:
# your code here

**4.** Make the first hour of the year at the four stations into a column of shape (4, 1), and print it.

In [13]:
# your code here

**5.** Make a table of the readings with one row per hour and one column per station, and print its last three rows.

In [14]:
# your code here

## Common errors

### ValueError: cannot reshape array of size 35040 into shape (4,newaxis,168)

In [15]:
weeks = readings.reshape(4, -1, 168)                       # weeks of 168 hours

ValueError: cannot reshape array of size 35040 into shape (4,newaxis,168)

`-1` can only stand for a whole number, and a year of 8,760 hours is 52 weeks and one day, so no
length makes the total come out. The message writes the `-1` as `newaxis`. Cut the readings to a
length that divides first: `readings[:, :52 * 168].reshape(4, 52, 168)` keeps 52 whole weeks.

### ValueError: can only specify one unknown dimension

In [16]:
readings.reshape(4, -1, -1)

ValueError: can only specify one unknown dimension

Two `-1` values could be filled in many ways, 365 by 24 or 24 by 365 or 73 by 120, so NumPy refuses
to choose, and every length but one has to be given.

### No error, and daily means of the wrong hours

In [17]:
right = readings.reshape(4, 365, 24).mean(axis=2)
wrong = readings.reshape(4, 24, 365).mean(axis=1)

print(right.shape, wrong.shape)
print(right[0, :3].round(2))
print(wrong[0, :3].round(2))

(4, 365) (4, 365)
[-0.7  -0.7  -0.73]
[7.95 7.97 7.99]


Both are 365 values per station, and only one is daily means. `(4, 24, 365)` filled its last
dimension first, so each of its "days" is 365 consecutive hours, and averaging across them mixes the
whole year together, which is why the wrong values sit close to Bergen's yearly mean of 8. Put the
dimension whose values are consecutive in memory, the hours of a day, last.

### No error, and a table that mixed stations and hours together

In [18]:
table = readings.reshape(8760, 4)

print(table[0])
print(readings.T[0])

[-3.6 -3.7 -3.7 -3.5]
[ -3.6  -3.5 -14.6  -6.7]


`reshape(8760, 4)` has the right shape, and reads Bergen's first four hours as the first row. The
first hour at the four stations is `readings.T[0]`. To swap rows and columns, transpose, and
`reshape` only to regroup values that are already in the right order.

### IndexError: tuple index out of range

In [19]:
oslo = readings[1]

print("hours:", oslo.shape[1])

IndexError: tuple index out of range

One station's row has one dimension, so its shape is a tuple of one number and `shape[1]` asks for a
second that is not there. Read the shape before indexing it: `oslo.shape` is `(8760,)`, and its
length is `oslo.shape[0]`, or `len(oslo)`.

## Recap

- Read `ndim`, `shape` and `size`, and tell a shape of `(8760,)` from `(8760, 1)`
- Regroup the readings into days with `reshape(4, 365, 24)`, putting the consecutive values last
- Let `-1` work out one length, and see why it fails when the total does not divide
- Flatten with `ravel` or `flatten`, and swap rows and columns with `.T`
- Add a dimension of length 1 with `np.newaxis` or `reshape(-1, 1)`

## What is next

The **Indexing and Slicing** notebook reads single values, rows, columns and ranges out of these
shapes, and shows when the piece you took still shares memory with the array.

---

&#8592; **Previous:** [dtype](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/03-dtype.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
